This notebook 
- requires 'reasons based dataset.csv' generated from '../Data Collection/reason-based-data.ipynb' 
- creates a 'complete data.csv' with columns ['Date', 'Fish', 'Fresh Type', 'Reason', 'S3_file_key'] from the images in the S3 bucket.
- creates a 'manual data.csv' from the images in the Manual Data folder with folder structure date/species/fish_type/img1.jpg
- then combines both manual data and s3 data files and stores in the 'complete data.csv' again. 
- this is created to store the metadata of our fish dataset with all the columns. This is useful to perform data analysis and generate reports without having to download the image data every time. We can also select specific dates and species, we are also saving time by skipping the already existing filenames.  

In [ ]:
import os
import shutil
import boto3
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta

# Set the environment variables
# use your AWS credentials insted of these
os.environ['AWS_ACCESS_KEY_ID'] = ''
os.environ['AWS_SECRET_ACCESS_KEY'] = ''

In [ ]:
# filepath = r"/content/drive/MyDrive/Sowmya /qZense Dataset/Misclassified_data.txt"
# misclassified_folder_path = r"/content/drive/MyDrive/Sowmya /qZense Dataset/Misclassified"
# with open(filepath, 'w') as f:
#     for file in os.listdir(misclassified_folder_path):
#         f.write(file+'\n')

In [ ]:
# Replace the path below with the actual path to your Misclassified_data.txt file
filepath = r"/content/drive/MyDrive/Sowmya /qZense Dataset/Misclassified_data.txt"
with open(filepath, 'r') as f:
    misclassified = [line[:-1] for line in f]

In [ ]:
print(len(misclassified))

1519


In [ ]:
class S3DataDownloader:
    '''
    This class initializes the S3DataDownloader instance, lists all objects in the specified S3 bucket, groups them by their date extracted from their key, and replaces misspelled folder names with correct ones. It also reads a CSV file containing reasons for classifying the fish as 'bad' and stores it in a DataFrame for later use.
    '''
    def __init__(self, bucket_name, download_path, reasons_filepath):
        """
        Initializes the S3DataDownloader instance.

        Args:
            bucket_name (str): The name of the S3 bucket.
            download_path (str): The local directory to download files to.
        """
        self.bucket_name = bucket_name
        self.download_path = download_path
        # Create an S3 client
        self.s3_client = boto3.client('s3')
        self.grouped_objects = self.group_objects_by_date()
        self.reasons_df = pd.read_csv(reasons_filepath)

    def list_s3_objects(self, bucket_name=None):
        """
        Lists all objects in the specified S3 bucket.

        Args:
            bucket_name (str): The name of the S3 bucket.

        Returns:
            list: A list of objects in the S3 bucket.
        """
        if bucket_name is None:
            bucket_name = self.bucket_name
        object_list = []
        # Use a paginator to iterate through all the objects in the bucket
        paginator = self.s3_client.get_paginator('list_objects_v2')
        page_iterator = paginator.paginate(Bucket=bucket_name)

        for page in page_iterator:
            # Get the list of objects in the current page
            objects = page.get('Contents', [])
            object_list.extend(objects)
        print("\nlist_s3_objects done")
        return object_list

    def group_objects_by_date(self):
        """
        Lists all objects in the S3 bucket & groups them by their last modified date.

        Returns:
            dict: A dictionary where keys are dates and values are lists of object keys.
        """
        objects = self.list_s3_objects()
        grouped_objects = {}
        for obj in objects:
            key = obj['Key']

            # extracting date from key
            date = (key.split('/')[-1])[:8]
            date = date[:4] + '-' + date[4:6] + '-' + date[6:8]

            if date in grouped_objects:
                grouped_objects[date].append(key)
            else:
                grouped_objects[date] = [key]

        # Sort the dictionary based on date values in the keys
        grouped_objects = dict(sorted(grouped_objects.items(),
                                      key=lambda item: item[0]))
        print("group_objects_by_date done\n")
        return grouped_objects

    def replace_misspelled_folder_names(self, species_name):
        misspelled_folders = {
            'Are' : ['Ar', 'Are'],
            'Basa' : ['Basa', 'Basaa'],
            'Barracuda' : ['Barcoda', 'Barkoda', 'Barracoda', 'Barracuda'],
            'Bolo' : ['Bolo', 'Bulo'],
            'Catla' : ['Katala', 'Katalaa', 'Katla'],
            'Croaker' : ['Kokor', 'Croaker', 'Silver croaker'],
            'Chara pona' : ['Chara pana'],
            'Demo' : ['Demo', 'Demo2', 'Test', 'Trial'],
            'Emperor' : ['Comprel', 'Emperor', 'Emporwel', 'Empowel',
                         'M perl', 'M preal'],
            'Hilsa' : ['Hilsa', 'Hilis', 'Hilisa'],
            'Lady' : ['Lady', 'Ledi'],
            'Malabar trevally' : ['Mabar tavili', 'Malbhot', 'Travely', 'Travaily',
                                  'Trvili', 'Trevally', 'Travelly', 'Giant trevally'],
            'Needle' : ['Needale', 'Nidal', 'Nidil'],
            'Parsi' : ['Parci'],
            'Pearl spot' : ['(bloch,', 'Bloch,', 'Bloch',
                            'Pearl spot', 'Pearls spot', 'Hols spot',
                            'Green chromide', 'Green chormide',
                            'Hals spot'],
            'Sea bass' : ['C boss', 'C boos', 'Siba'],
            'Shol' : ['Sholo'],
            'Snapper' : ['Sinper', 'Sniper'],
            'White snapar' : ['White snapper'],
        }

        for key, misspellings in misspelled_folders.items():
            if species_name in misspellings:
                return key
        return species_name

In [ ]:
# Example usage
bucket_name='fish-data-collection-v2'
csv_filepath = '/content/drive/MyDrive/Sowmya /Data Collection/v5/complete data.csv'
reasons_filepath = '/content/drive/MyDrive/Sowmya /reasons Dataset/reasons_dataset.csv'

data_downloader=S3DataDownloader(bucket_name, csv_filepath, reasons_filepath)


list_s3_objects done
group_objects_by_date done



In [ ]:
self = data_downloader

In [ ]:
self.grouped_objects.keys()

dict_keys(['2023-05-18', '2023-05-19', '2023-05-20', '2023-05-23', '2023-05-25', '2023-05-26', '2023-05-27', '2023-05-28', '2023-05-30', '2023-06-01', '2023-06-02', '2023-06-03', '2023-06-04', '2023-06-06', '2023-06-08', '2023-06-09', '2023-06-10', '2023-06-11', '2023-06-12', '2023-06-13', '2023-06-15', '2023-06-16', '2023-06-17', '2023-06-18', '2023-06-20', '2023-06-22', '2023-06-23', '2023-06-24', '2023-06-25', '2023-06-27', '2023-06-29', '2023-06-30', '2023-07-01', '2023-07-02', '2023-07-04', '2023-07-06', '2023-07-07', '2023-07-08', '2023-07-09', '2023-07-11', '2023-07-13', '2023-07-14', '2023-07-15', '2023-07-16', '2023-07-18', '2023-07-20', '2023-07-22', '2023-07-23', '2023-07-25', '2023-07-27', '2023-07-28', '2023-07-29', '2023-07-30', '2023-08-01', '2023-08-03', '2023-08-04', '2023-08-05', '2023-08-06', '2023-08-08', '2023-08-09', '2023-08-10', '2023-08-11', '2023-08-12', '2023-08-13', '2023-08-15', '2023-08-17', '2023-08-18', '2023-08-19', '2023-08-20', '2023-08-22', '2023-08-

In [ ]:
def create_dataframe(self, date, keys):
    '''
    This function downloads the images from the S3 bucket and creates a dataframe from the downloaded images
    '''
    
    if os.path.exists(self.download_path):
        df = pd.read_csv(self.download_path)
    else:
        columns=['Date', 'Filename', 'Species', 'Class', 'Reason',
                  'S3_file_key', 'BucketName']
        df = pd.DataFrame(columns=columns)

    stop = False

    # Iterate through the keys and extract fish name and fresh type from the image name and add it to the dataframe
    existing = set(df['Filename'].values)
    reason_lookup = (self.reasons_df.set_index("Filename")["description"].to_dict())

    for key in tqdm(keys, desc=f'Loading {date} data'):
        d = {}
        # Extract fish name and fresh type from the image name
        image_name = key.split('/')[-1]
        if image_name in existing:
            continue

        try:
            fish_name, fresh_type = image_name.split('_')[-2:]
        except:
            continue
        fresh_type = fresh_type.split('.')[0]
        fish_name = fish_name.capitalize()
        fresh_type = fresh_type.capitalize()
        fish_name = fish_name.strip()
        fresh_type = fresh_type.strip()
        fish_name = self.replace_misspelled_folder_names(fish_name)

        d['Date'] = date
        d['Filename'] = image_name
        d['Species'] = fish_name
        d['Class'] = fresh_type

        d['Reason'] = reason_lookup.get(image_name)
        d['S3_file_key'] = key

        if image_name in misclassified:
            d['Class'] = 'Misclassified'

        d['BucketName'] = self.bucket_name

        new_df = pd.DataFrame([d])
        df = pd.concat([df, new_df], ignore_index=True)
        df.to_csv(self.download_path, index=False)


def download_daily_data(self):
    for date, keys in self.grouped_objects.items():
        # if date<'2025-01-18':
        #     continue
        create_dataframe(self, date, keys)

In [ ]:
self.download_path = r'..\generated-files\complete data.csv'

In [ ]:
download_daily_data(self)

Loading 2_ma-ck-er data: 100%|██████████| 2/2 [00:00<00:00, 259.44it/s]


In [ ]:
# loaded_df = pd.read_csv('complete data.csv')
loaded_df = pd.read_csv(self.download_path)
loaded_df

,Date,Filename,Species,Class,Reason,S3_file_key,BucketName
0,2023-05-18,20230518085305657_mackerel_ok.jpeg,Mackerel,Ok,NaN,mackerel/ok/20230518085305657_mackerel_ok.jpeg,fish-data-collection-v2
1,2023-05-18,20230518084706566_rohu_good.jpeg,Rohu,Good,NaN,rohu/good/20230518084706566_rohu_good.jpeg,fish-data-collection-v2
2,2023-05-19,20230519071719731_mackerel_good.jpeg,Mackerel,Good,NaN,mackerel/good/20230519071719731_mackerel_good....,fish-data-collection-v2
3,2023-05-20,20230520121600220_indian salmon _good.jpeg,Indian salmon,Good,NaN,Indian salmon /good/20230520121600220_indian s...,fish-data-collection-v2
4,2023-05-20,20230520121628822_indian salmon _good.jpeg,Indian salmon,Good,NaN,Indian salmon /good/20230520121628822_indian s...,fish-data-collection-v2
...,...,...,...,...,...,...,...
89183,2025-01-28,20250128113440816_pink perch_bad.jpeg,Pink perch,Bad,Softness,pink perch/bad/20250128113440816_pink perch_ba...,fish-data-collection-v2
89184,2025-01-28,20250128113451297_pink perch_bad.jpeg,Pink perch,Bad,Softness,pink perch/bad/20250128113451297_pink perch_ba...,fish-data-collection-v2
89185,2025-01-28,20250128113459764_pink perch_bad.jpeg,Pink perch,Bad,Softness,pink perch/bad/20250128113459764_pink perch_ba...,fish-data-collection-v2
89186,2025-01-28,20250128113508768_pink perch_bad.jpeg,Pink perch,Bad,Softness,pink perch/bad/20250128113508768_pink perch_ba...,fish-data-collection-v2


In [ ]:
self.download_path

'/content/drive/MyDrive/Sowmya /Data Collection/v5/complete data.csv'

In [ ]:
loaded_df['Reason'].value_counts()

,count
Reason,
Softness,19480
Manual_data,6211
Size,819
Cuts and Damage,632
Red Head in Prawns,473
"Softness, Cuts and Damage",460
"Cuts and Damage, Softness",442
"Softness, Size",213
"Size, Softness",192


# Manual Data details

In [ ]:
'''
This script creates a dataframe from the manual data present in the folder structure and saves it to a CSV file. The folder structure is expected to be in the format: date/species/fresh_type/filename. If the CSV file already exists, it will be loaded and updated with any new files found in the folder structure. 

The resulting dataframe will contain columns for Date, Filename, Species, Class, and Reason.
'''

# path to the folder containing the manual data 
folder_path = 'Manual Data' 

# path to the csv file where the dataframe will be saved
csv_path = r'..\generated-files\manual data.csv' 


if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    columns=['Date', 'Filename', 'Species', 'Class', 'Reason']
    df = pd.DataFrame(columns=columns)

for date in os.listdir(folder_path):
    date_folder_path = os.path.join(folder_path, date)
    if not os.path.isdir(date_folder_path):
        continue
    for species in tqdm(os.listdir(date_folder_path), desc=f'{date}'):
        species_folder_path = os.path.join(date_folder_path, species)
        for fresh_type in os.listdir(species_folder_path):
            fresh_type_folder_path = os.path.join(species_folder_path, fresh_type)
            for filename in os.listdir(fresh_type_folder_path):
                file_path = os.path.join(fresh_type_folder_path, filename)
                file_dict = {'Date':date,
                             'Filename':filename,
                             'Species':species,
                             'Class':fresh_type,
                             'Reason':'Manual_data'}
                if filename in ['Single', 'Group']:
                    for img in os.listdir(file_path):
                        if img in df['Filename'].values:
                            continue
                        file_dict['Filename']=img
                        new_df = pd.DataFrame([file_dict])
                        df = pd.concat([df, new_df], ignore_index=True)
                        df.to_csv(csv_path, index=False)
                else:
                    if filename in df['Filename'].values:
                        continue
                    new_df = pd.DataFrame([file_dict])
                    df = pd.concat([df, new_df], ignore_index=True)
                    df.to_csv(csv_path, index=False)

df

2024-08-27: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]
2024-08-29: 0it [00:00, ?it/s]
2025-01-09: 100%|██████████| 3/3 [00:03<00:00,  1.15s/it]


,Date,Filename,Species,Class,Reason
0,2023-06-08,IMG20230608080643.jpg,Mackerel,Good,Manual_data
1,2023-06-08,IMG20230608080649.jpg,Mackerel,Good,Manual_data
2,2023-06-08,IMG20230608080700.jpg,Mackerel,Good,Manual_data
3,2023-06-08,IMG20230608080655.jpg,Mackerel,Good,Manual_data
4,2023-06-08,IMG20230608080717.jpg,Mackerel,Good,Manual_data
...,...,...,...,...,...
6206,2025-01-09,20250109_130020.jpg,Mackerel,Good,Manual_data
6207,2025-01-09,20250109_130046.jpg,Mackerel,Good,Manual_data
6208,2025-01-09,20250109_130051.jpg,Mackerel,Good,Manual_data
6209,2025-01-09,20250109_125909.jpg,Mackerel,Good,Manual_data


# Combine Manual Data and S3 data details

In [ ]:
total_data_df = pd.read_csv(r'..\generated-files\complete data.csv')
total_data_df.shape

(88681, 7)

In [ ]:
manual_data_df = pd.read_csv(r'..\generated-files\manual data.csv')
manual_data_df.shape

(6211, 5)

In [ ]:
manual_data_df['Species'].unique()

array(['Mackerel', 'Sardine', 'White prawns'], dtype=object)

In [ ]:
new_df = pd.concat([total_data_df, manual_data_df], ignore_index=True)
new_df

,Date,Filename,Species,Class,Reason,S3_file_key,BucketName
0,2023-05-18,20230518085305657_mackerel_ok.jpeg,Mackerel,Ok,NaN,mackerel/ok/20230518085305657_mackerel_ok.jpeg,fish-data-collection-v2
1,2023-05-18,20230518084706566_rohu_good.jpeg,Rohu,Good,NaN,rohu/good/20230518084706566_rohu_good.jpeg,fish-data-collection-v2
2,2023-05-19,20230519071719731_mackerel_good.jpeg,Mackerel,Good,NaN,mackerel/good/20230519071719731_mackerel_good....,fish-data-collection-v2
3,2023-05-20,20230520121600220_indian salmon _good.jpeg,Indian salmon,Good,NaN,Indian salmon /good/20230520121600220_indian s...,fish-data-collection-v2
4,2023-05-20,20230520121628822_indian salmon _good.jpeg,Indian salmon,Good,NaN,Indian salmon /good/20230520121628822_indian s...,fish-data-collection-v2
...,...,...,...,...,...,...,...
94887,2025-01-09,20250109_130020.jpg,Mackerel,Good,Manual_data,NaN,NaN
94888,2025-01-09,20250109_130046.jpg,Mackerel,Good,Manual_data,NaN,NaN
94889,2025-01-09,20250109_130051.jpg,Mackerel,Good,Manual_data,NaN,NaN
94890,2025-01-09,20250109_125909.jpg,Mackerel,Good,Manual_data,NaN,NaN


In [ ]:
new_df.drop_duplicates(inplace=True)
new_df.shape

(88681, 7)

In [ ]:
new_df.to_csv(r'..\generated-files\complete data.csv', index=False)

In [ ]:
new_df = pd.read_csv(r'..\generated-files\complete data.csv')
new_df

,Date,Filename,Species,Class,Reason,S3_file_key,BucketName
0,2023-05-18,20230518085305657_mackerel_ok.jpeg,Mackerel,Ok,NaN,mackerel/ok/20230518085305657_mackerel_ok.jpeg,fish-data-collection-v2
1,2023-05-18,20230518084706566_rohu_good.jpeg,Rohu,Good,NaN,rohu/good/20230518084706566_rohu_good.jpeg,fish-data-collection-v2
2,2023-05-19,20230519071719731_mackerel_good.jpeg,Mackerel,Good,NaN,mackerel/good/20230519071719731_mackerel_good....,fish-data-collection-v2
3,2023-05-20,20230520121600220_indian salmon _good.jpeg,Indian salmon,Good,NaN,Indian salmon /good/20230520121600220_indian s...,fish-data-collection-v2
4,2023-05-20,20230520121628822_indian salmon _good.jpeg,Indian salmon,Good,NaN,Indian salmon /good/20230520121628822_indian s...,fish-data-collection-v2
...,...,...,...,...,...,...,...
88676,2025-01-21,20250121110117647_sardine_good.jpeg,Sardine,Good,NaN,sardine/good/20250121110117647_sardine_good.jpeg,fish-data-collection-v2
88677,2025-01-21,20250121110128356_sardine_good.jpeg,Sardine,Good,NaN,sardine/good/20250121110128356_sardine_good.jpeg,fish-data-collection-v2
88678,2025-01-21,20250121110140342_sardine_good.jpeg,Sardine,Good,NaN,sardine/good/20250121110140342_sardine_good.jpeg,fish-data-collection-v2
88679,2025-01-21,20250121110239996_sardine_good.jpeg,Sardine,Good,NaN,sardine/good/20250121110239996_sardine_good.jpeg,fish-data-collection-v2
